# Notebook 01 — SFT Dataset A (Crownelius / Opus Reasoning)
**Assignment 04 | Track 2, Option D | NLP with Deep Learning — IBA**

**Model:** `Qwen/Qwen3-0.6B`  
**Dataset:** `Crownelius/Opus-4.6-Reasoning-3300x` — 2,160 rows with `problem`, `thinking`, `solution`, `difficulty`, `category`

**Ablation design (5 trials):**
| Trial | Data % | LoRA Rank | Target Modules | LR | Batch | Grad Accum | Epochs | Purpose |
|-------|--------|-----------|----------------|----|-------|------------|--------|---------|
| T1 | 25% | 16 | q,k,v,o | 2e-4 | 2 | 4 | 1 | Data efficiency — low |
| T2 | 50% | 16 | q,k,v,o | 2e-4 | 2 | 4 | 1 | Data efficiency — mid |
| T3 | 100% | 16 | q,k,v,o | 2e-4 | 2 | 4 | 1 | Data efficiency — full |
| T4 | 100% | 32 | q,k,v,o,gate,up,down | 1e-4 | 4 | 4 | 2 | LoRA capacity — wider |
| T5 | 100% | 64 | q,k,v,o | 5e-5 | 2 | 8 | 2 | LoRA capacity — deeper |

**Run this on:** Any CUDA GPU machine (local 3090, Kaggle T4, Colab)  
**Prerequisite:** Run `00_baseline.ipynb` first to generate `prompts.json` and `gold_answers.json`

In [1]:
# ── Install dependencies ──────────────────────────────────────────────────────
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

# Unsloth: try the CUDA 12.4 / Torch 2.6 build first, then fall back to the package release.
try:
    pip_install("unsloth[cu124-torch260] @ git+https://github.com/unslothai/unsloth.git")
except subprocess.CalledProcessError:
    pip_install("unsloth")

pip_install("trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "huggingface_hub", "python-dotenv", "openai")

# Keep the Windows CUDA stack aligned with Torch 2.6 / CUDA 12.4.
pip_install("torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124")
pip_install("--force-reinstall", "--no-deps", "triton-windows==3.2.0.post21", "torchao==0.16.0")
pip_install("pyarrow==21.0.0", "fsspec==2025.9.0")

In [2]:
import os, json, time, re, torch, pandas as pd
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# ── CONFIG ────────────────────────────────────────────────────────────────────
HF_TOKEN    = os.environ["HF_TOKEN"]   # loaded from .env
BASE_MODEL  = "Qwen/Qwen3-0.6B"
DATASET_ID  = "Crownelius/Opus-4.6-Reasoning-3300x"
JUDGE_MODEL = "moonshotai/Kimi-K2-Instruct:novita"
JUDGE_CLIENT = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN,
)
MAX_SEQ_LEN = 2048
OUTPUT_DIR  = "./adapters_datasetA"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU available: True
GPU: NVIDIA GeForce RTX 3090 | VRAM: 25.8 GB


## Step 1 — Load Dataset and Verify Columns

In [3]:
ds = load_dataset(DATASET_ID, split="train")
print(f"Dataset size: {len(ds)} rows")
print(f"Columns: {ds.features}")
print("\nSample row:")
sample = ds[0]
for k, v in sample.items():
    print(f"  {k}: {str(v)[:120]}")

Dataset size: 2160 rows
Columns: {'id': Value('string'), 'problem': Value('string'), 'thinking': Value('string'), 'solution': Value('string'), 'difficulty': Value('string'), 'category': Value('string'), 'timestamp': Value('string'), 'hash': Value('string')}

Sample row:
  id: drive_minimax_m2.1_questions_70109
  problem: Your solution must read input from standard input (input()), write output to standard output (print()).
Do not include a
  thinking: Looking at what was provided, this appears to be an incomplete or truncated problem statement. The provided text contain
  solution: I notice you've provided instructions for how you'd like a solution formatted, but you haven't actually included a probl
  difficulty: medium
  category: math
  timestamp: 2026-02-10T04:41:12.158240+00:00
  hash: b60b4c8dd167424d


## Step 2 — Format Dataset with Chat Template

We include the `thinking` field in the assistant turn using `<think>` tags.
This is the key structural advantage of Dataset A — explicit chain-of-thought supervision.

In [4]:
def get_tokenizer():
    """Load tokenizer once to format examples."""
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    return tok

tokenizer_for_format = get_tokenizer()

def format_row(row):
    """Format a Crownelius row into a chat-template string.
    Includes thinking trace for CoT supervision.
    """
    thinking = row.get("thinking", "").strip()
    solution = row.get("solution", "").strip()

    if thinking:
        assistant_content = f"<think>{thinking}</think>\n{solution}"
    else:
        assistant_content = solution

    messages = [
        {"role": "system",    "content": "You are a helpful reasoning assistant. Think step by step."},
        {"role": "user",      "content": row["problem"].strip()},
        {"role": "assistant", "content": assistant_content},
    ]
    return tokenizer_for_format.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

ds_formatted = ds.map(lambda row: {"text": format_row(row)})
print("Sample formatted text (first 600 chars):")
print(ds_formatted[0]["text"][:600])

Sample formatted text (first 600 chars):
<|im_start|>system
You are a helpful reasoning assistant. Think step by step.<|im_end|>
<|im_start|>user
Your solution must read input from standard input (input()), write output to standard output (print()).
Do not include any debug prints or additional output.<|im_end|>
<|im_start|>assistant
<think>
Looking at what was provided, this appears to be an incomplete or truncated problem statement. The provided text contains input/output format specifications, but these are only fragments of a larger problem. Without the full problem description — including the complete problem statement, all cons


## Step 3 — Load Prompts and Gold Answers for Evaluation

In [5]:
# Load from notebook 00 outputs — same working directory on Kaggle
with open("prompts.json") as f:
    PROMPTS = json.load(f)
with open("gold_answers.json") as f:
    gold_answers = json.load(f)

print(f"Loaded {len(PROMPTS)} prompts and {len(gold_answers)} gold answers.")

JUDGE_PROMPT_TEMPLATE = """\
You are an expert evaluator of reasoning quality.

QUESTION:
{question}

REFERENCE ANSWER (Gold):
{gold}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate response on REASONING QUALITY:
5 = Correct answer, clear step-by-step reasoning, no errors
4 = Correct answer, reasoning mostly clear with minor gaps
3 = Partially correct or reasoning has notable gaps
2 = Wrong answer but shows some reasoning attempt
1 = Wrong answer, no meaningful reasoning

Respond in this exact format (nothing else):
SCORE: <number 1-5>
REASON: <one sentence justification>"""


def _parse_judge_output(out):
    score_match = re.search(r"SCORE\s*:\s*([1-5])", out, re.IGNORECASE)
    reason_match = re.search(r"REASON\s*:\s*(.+)", out, re.IGNORECASE | re.DOTALL)
    if not score_match:
        raise ValueError(f"Judge response did not include SCORE 1-5. Raw output: {out!r}")
    reason = reason_match.group(1).strip() if reason_match else out.strip()
    return int(score_match.group(1)), reason


def judge_response(question, gold, response):
    prompt = JUDGE_PROMPT_TEMPLATE.format(question=question, gold=gold, response=response)
    last_error = None
    for attempt in range(3):
        try:
            completion = JUDGE_CLIENT.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=120,
                temperature=0.0,
            )
            out = completion.choices[0].message.content.strip()
            return _parse_judge_output(out)
        except Exception as error:
            last_error = error
            time.sleep(2 * (attempt + 1))
    return 0, f"ERROR after retries: {last_error}"



def _apply_chat_template(tokenizer, messages):
    """Handles BatchEncoding (transformers 4.50+) and plain tensor (older)."""
    encoded = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    return encoded["input_ids"] if not isinstance(encoded, torch.Tensor) else encoded


def eval_model_on_prompts(model, tokenizer, trial_name):
    """Run fine-tuned model on all 10 prompts and get judge scores."""
    FastLanguageModel.for_inference(model)
    records = []
    for p in PROMPTS:
        messages = [
            {"role": "system",  "content": "You are a helpful reasoning assistant. Think step by step."},
            {"role": "user",    "content": p["prompt"]}
        ]
        input_ids = _apply_chat_template(tokenizer, messages).to(model.device)
        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=512, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id
            )
        response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        score, reason = judge_response(p["prompt"], gold_answers[p["id"]], response)
        records.append({
            "trial": trial_name, "prompt_id": p["id"], "type": p["type"],
            "response": response, "score": score, "reason": reason
        })
        print(f"  [{trial_name}] {p['id']}: score={score}")
        time.sleep(0.5)
    return records

Loaded 10 prompts and 10 gold answers.


## Step 4 — Trial Loop (5 Trials)

Each trial: train → save adapter → evaluate on 10 prompts → judge score

In [7]:
TRIAL_CONFIGS = [
    {
        "name": "T1", "data_pct": 0.25, "rank": 16,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
        "purpose": "Data efficiency — 25%"
    },
    {
        "name": "T2", "data_pct": 0.50, "rank": 16,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
        "purpose": "Data efficiency — 50%"
    },
    {
        "name": "T3", "data_pct": 1.00, "rank": 16,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 2e-4, "batch": 2, "grad_accum": 4, "epochs": 1,
        "purpose": "Data efficiency — 100%"
    },
    {
        "name": "T4", "data_pct": 1.00, "rank": 32,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        "lr": 1e-4, "batch": 4, "grad_accum": 4, "epochs": 2,
        "purpose": "LoRA capacity — rank 32, wider modules"
    },
    {
        "name": "T5", "data_pct": 1.00, "rank": 64,
        "target_modules": ["q_proj","k_proj","v_proj","o_proj"],
        "lr": 5e-5, "batch": 2, "grad_accum": 8, "epochs": 2,
        "purpose": "LoRA capacity — rank 64, deeper"
    },
]

# fp16/bf16 — only set when actually on GPU; CPU training uses full fp32
use_fp16 = torch.cuda.is_available() and not torch.cuda.is_bf16_supported()
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"fp16={use_fp16} | bf16={use_bf16}")

all_records = []
trial_summary = []

for cfg in TRIAL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Trial {cfg['name']}: {cfg['purpose']}")
    print(f"  data={cfg['data_pct']*100:.0f}%, rank={cfg['rank']}, lr={cfg['lr']}, "
          f"batch={cfg['batch']}, grad_accum={cfg['grad_accum']}, epochs={cfg['epochs']}")
    print('='*60)

    n = int(len(ds_formatted) * cfg["data_pct"])
    ds_train = ds_formatted.select(range(n))
    split = ds_train.train_test_split(test_size=0.1, seed=42)
    train_ds = split["train"]
    eval_ds  = split["test"]
    print(f"  Train: {len(train_ds)} | Eval: {len(eval_ds)}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True, token=HF_TOKEN
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=cfg["rank"],
        target_modules=cfg["target_modules"],
        lora_alpha=cfg["rank"],
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing=True,
    )

    adapter_path = os.path.join(OUTPUT_DIR, cfg["name"])

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        args=SFTConfig(
            output_dir=adapter_path,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            per_device_train_batch_size=cfg["batch"],
            gradient_accumulation_steps=cfg["grad_accum"],
            learning_rate=cfg["lr"],
            num_train_epochs=cfg["epochs"],
            eval_strategy="epoch",
            save_strategy="epoch",
            # load_best_model_at_end NOT used — incompatible with Unsloth 4-bit quantized models
            fp16=use_fp16,
            bf16=use_bf16,
            logging_steps=10,
            disable_tqdm=True,
            report_to="none",
        ),
    )
    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass

    train_result = trainer.train()
    val_loss = trainer.evaluate()["eval_loss"]

    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"  Adapter saved to: {adapter_path}")
    print(f"  Train loss: {train_result.training_loss:.4f} | Val loss: {val_loss:.4f}")

    print("  Evaluating on 10 baseline prompts...")
    records = eval_model_on_prompts(model, tokenizer, cfg["name"])
    for r in records:
        r.update({"model": BASE_MODEL, "stage": "sft_datasetA",
                  "data_pct": cfg["data_pct"], "rank": cfg["rank"],
                  "lr": cfg["lr"], "val_loss": val_loss})
    all_records.extend(records)

    avg_score = sum(r["score"] for r in records) / len(records)
    trial_summary.append({
        "trial": cfg["name"], "purpose": cfg["purpose"],
        "data_pct": f"{cfg['data_pct']*100:.0f}%",
        "rank": cfg["rank"], "lr": cfg["lr"],
        "batch": cfg["batch"], "epochs": cfg["epochs"],
        "val_loss": round(val_loss, 4),
        "avg_judge_score": round(avg_score, 2),
        "adapter_path": adapter_path,
    })
    print(f"  Avg judge score: {avg_score:.2f}")

    del model, trainer
    torch.cuda.empty_cache()

print("\nAll 5 trials complete.")

fp16=False | bf16=True

Trial T1: Data efficiency — 25%
  data=25%, rank=16, lr=0.0002, batch=2, grad_accum=4, epochs=1
  Train: 486 | Eval: 54
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-0.6b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Tokenizing ["text"]:   0%|          | 0/486 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/54 [00:00<?, ? examples/s]

...

  [T5] M3: score=5


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [T5] M4: score=2


Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [T5] M5: score=5
  Avg judge score: 3.90

All 5 trials complete.


## Step 5 — Select Best Trial and Save Results

In [8]:
df_summary = pd.DataFrame(trial_summary)
print("\n── Trial Results ──")
print(df_summary[["trial","purpose","data_pct","rank","lr","epochs","val_loss","avg_judge_score"]].to_string(index=False))

# Select best: highest avg_judge_score, tie-break on lowest val_loss
best = df_summary.sort_values(
    ["avg_judge_score", "val_loss"], ascending=[False, True]
).iloc[0]

print(f"\n── Best Trial: {best['trial']} ──")
print(f"  Purpose:         {best['purpose']}")
print(f"  Avg judge score: {best['avg_judge_score']}")
print(f"  Val loss:        {best['val_loss']}")
print(f"  Adapter saved:   {best['adapter_path']}")

# Save
df_summary.to_csv("trial_results_datasetA.csv", index=False)
pd.DataFrame(all_records).to_csv("all_scores_datasetA.csv", index=False)
with open("best_trial_datasetA.json", "w") as f:
    json.dump(best.to_dict(), f, indent=2)

print("\nSaved: trial_results_datasetA.csv, all_scores_datasetA.csv, best_trial_datasetA.json")


── Trial Results ──
trial                                purpose data_pct  rank      lr  epochs  val_loss  avg_judge_score
   T1                  Data efficiency — 25%      25%    16 0.00020       1    1.3807              4.1
   T2                  Data efficiency — 50%      50%    16 0.00020       1    1.3521              3.7
   T3                 Data efficiency — 100%     100%    16 0.00020       1    1.2112              3.6
   T4 LoRA capacity — rank 32, wider modules     100%    32 0.00010       2    1.1449              2.5
   T5        LoRA capacity — rank 64, deeper     100%    64 0.00005       2    1.2260              3.9

── Best Trial: T1 ──
  Purpose:         Data efficiency — 25%
  Avg judge score: 4.1
  Val loss:        1.3807
  Adapter saved:   ./adapters_datasetA\T1

Saved: trial_results_datasetA.csv, all_scores_datasetA.csv, best_trial_datasetA.json
